In [1]:
%cd /home/parthgandhi/Projects/SwingBot/swingbot/src

/home/parthgandhi/Projects/SwingBot/swingbot/src


In [2]:
import logging

import polars as pl
import polars.selectors as cs

from config.computation.compute import ComputeConfig
from config.computation.indicator import IndicatorConfig
from utils import setup_logger
from config.base import StorageConfig

logger = logging.getLogger(__name__)

setup_logger()

In [3]:
end_date = "2026-03-13"
save_path = StorageConfig().store_root(ComputeConfig.FOLDER_NAME, end_date)

# Scanner DataFrame

In [6]:
res = (
    pl.scan_csv(save_path / ComputeConfig.FILTER_RESULT_PATH)
    .select(
        [
            "symbol",
            "pct_gain_prev_1",
            "pct_gain_prev_5",
            "pct_gain_prev_21",
            "pct_gain_prev_63",
            "pct_gain_prev_126",
            "adr_pct_20",
            "adr_filter_flag",
            "pullback_filter_flag",
            "mid_down_streak",
            "near_ema_9",
            "near_ema_21",
            "near_sma_50",
            "macro_economic_sector",
            "sector",
            "industry",
            "basic_industry",
            "market_cap_cr",
        ]
    )
    .rename(
        {
            f"pct_gain_prev_{i}": f"{value}"
            for i, value in IndicatorConfig.LOOKBACK_RETURN_PCT.items()
        }
    )
    .collect()
)

res = (
    res.lazy()
    .rename({i: i.replace("_", " ").upper() for i in res.columns})
    .with_columns(cs.float().round(2))
    .collect()
)
res.select(cs.float()).columns

['1D', '1W', '1M', '3M', '6M', 'ADR PCT 20', 'MARKET CAP CR']